In [12]:
symbols = [
    'RELIANCE',
    'HDFCBANK',
    'BHARTIARTL',
    'SBIN',
    'ICICIBANK',
    'TCS',
    'BAJFINANCE',
    'LT',
    'LICI',
    'HINDUNILVR'
]

In [13]:
import duckdb
import pprint

# Connect to local DuckDB file
db_path = "C:\\Users\\Keerti A M\\Documents\\New folder\\data\\india_market (2).duckdb"
conn = duckdb.connect(db_path)

# Get all table names
tables_df = conn.execute("SHOW TABLES").fetchdf()
table_names = tables_df["name"].tolist()

all_column_names = {}

# Get column names for each table
for table_name in table_names:
    columns_desc = conn.execute(
        f"SELECT * FROM {table_name} LIMIT 0"
    ).description

    column_names = [desc[0] for desc in columns_desc]

    all_column_names[table_name] = column_names

# Print schema
pprint.pprint(all_column_names)

# Close connection


{'companies': ['symbol',
               'company_name',
               'sector',
               'industry',
               'isin',
               'listing_date',
               'cap_category',
               'avg_mcap_cr',
               'exchange',
               'bse_code'],
 'company_betas': ['symbol',
                   'regression_beta',
                   'blume_beta',
                   'damodaran_beta',
                   'blended_beta',
                   'r_squared',
                   'n_obs',
                   'computed_date'],
 'corporate_actions': ['symbol', 'date', 'action_type', 'value', 'remarks'],
 'fetched_dates': ['date'],
 'financials': ['symbol',
                'period',
                'year',
                'quarter',
                'revenue',
                'net_profit',
                'ebitda',
                'eps',
                'assets',
                'liabilities',
                'equity',
                'debt',
                'operating_cash_

In [14]:
import pandas as pd

# Fetch company details
companies_query = f"""
SELECT symbol, company_name, sector, industry, cap_category, avg_mcap_cr
FROM companies
WHERE symbol IN ({','.join([f"'{s}'" for s in symbols])})
"""
companies_df = conn.execute(companies_query).fetchdf()

# Fetch price data
prices_query = f"""
SELECT symbol, date, open, high, low, close, volume
FROM prices
WHERE symbol IN ({','.join([f"'{s}'" for s in symbols])})
ORDER BY symbol, date
"""
prices_df = conn.execute(prices_query).fetchdf()

# Convert date column to datetime objects
prices_df['date'] = pd.to_datetime(prices_df['date'])

print("Prices fetched. Shape:", prices_df.shape)
print("Companies fetched. Shape:", companies_df.shape)
print("Merging with companies will happen after price adjustment in the next cells.")

Prices fetched. Shape: (54796, 7)
Companies fetched. Shape: (10, 6)
Merging with companies will happen after price adjustment in the next cells.


In [15]:
# ── Inspect corporate actions ────────────────────────────────────────────
# The prices table has RAW prices. Stock splits and bonus issues cause sudden
# price halving/thirding that looks like a massive crash to the model but
# isn't real. We need to adjust prices backward using corporate actions.

corp_query = f"""
SELECT symbol, date, action_type, value, remarks
FROM corporate_actions
WHERE symbol IN ({','.join([f"'{s}'" for s in symbols])})
ORDER BY symbol, date
"""
corp_df = conn.execute(corp_query).fetchdf()
corp_df['date'] = pd.to_datetime(corp_df['date'])

print(f"Total corporate actions: {len(corp_df)}")
print(f"Unique action types: {sorted(corp_df['action_type'].unique())}")
print()
display(corp_df)

Total corporate actions: 356
Unique action types: ['dividend', 'split']



,symbol,date,action_type,value,remarks
0,BAJFINANCE,2003-07-10,dividend,0.043721,yfinance: Dividend ₹0.04/share
1,BAJFINANCE,2004-07-15,dividend,0.058295,yfinance: Dividend ₹0.06/share
2,BAJFINANCE,2005-06-29,dividend,0.072869,yfinance: Dividend ₹0.07/share
3,BAJFINANCE,2006-06-29,dividend,0.038863,yfinance: Dividend ₹0.04/share
4,BAJFINANCE,2007-06-28,dividend,0.029147,yfinance: Dividend ₹0.03/share
...,...,...,...,...,...
351,TCS,2025-01-17,dividend,76.000000,yfinance: Dividend ₹76.00/share
352,TCS,2025-06-04,dividend,30.000000,yfinance: Dividend ₹30.00/share
353,TCS,2025-07-16,dividend,11.000000,yfinance: Dividend ₹11.00/share
354,TCS,2025-10-15,dividend,11.000000,yfinance: Dividend ₹11.00/share


In [16]:
# ── Apply backward price adjustment ─────────────────────────────────────
# Goal: replace raw open/high/low/close with split-and-dividend-adjusted prices
# so that log-returns are clean economic signals, not corporate-action artifacts.
#
# How it works (working BACKWARD from most recent date):
#   Split 2:1  → all prices BEFORE that date are multiplied by 0.5
#   Bonus 1:1  → same as a 2:1 split — prices before multiplied by 0.5
#   Bonus 2:1  → 2 new shares per 1 held → prices before multiplied by 1/3
#   Dividend ₹d on ex-date close ₹P → factor = (P - d) / P applied to all prices before
#
# The most recent adjusted price always equals the raw price (no compounding forward),
# so test.py shows the real current market price.

def parse_ratio(value, action_label):
    """Parse ratio strings like '1:1', '2:1', or plain floats."""
    try:
        if isinstance(value, str) and ':' in value:
            a, b = value.split(':', 1)
            return float(a.strip()) / float(b.strip())
        return float(value)
    except (ValueError, ZeroDivisionError, AttributeError):
        print(f"  WARNING: cannot parse {action_label} value '{value}' — skipping")
        return None


def compute_adj_prices(prices_df, corp_df):
    prices_df = prices_df.copy().sort_values(['symbol', 'date']).reset_index(drop=True)

    # Start with copies of raw prices
    for col in ['open', 'high', 'low', 'close']:
        prices_df[f'adj_{col}'] = prices_df[col].astype(float)

    skipped = []

    for symbol in prices_df['symbol'].unique():
        sym_mask  = prices_df['symbol'] == symbol
        sym_corp  = corp_df[corp_df['symbol'] == symbol].sort_values('date', ascending=False)

        for _, row in sym_corp.iterrows():
            adate = row['date']
            atype = str(row['action_type']).upper().strip()
            val   = row['value']

            before = sym_mask & (prices_df['date'] < adate)
            if before.sum() == 0:
                continue

            factor = None

            if 'SPLIT' in atype:
                ratio = parse_ratio(val, f'{symbol} SPLIT {adate.date()}')
                if ratio and ratio > 0:
                    factor = 1.0 / ratio   # 2:1 split → historical prices × 0.5

            elif 'BONUS' in atype:
                # Bonus 1:1 → 1 extra share per 1 held → total shares double → price halves
                ratio = parse_ratio(val, f'{symbol} BONUS {adate.date()}')
                if ratio and ratio > 0:
                    factor = 1.0 / (1.0 + ratio)

            elif 'DIV' in atype or 'DIVIDEND' in atype:
                try:
                    div_amt = float(val)
                    ex_rows = prices_df[sym_mask & (prices_df['date'] == adate)]
                    if len(ex_rows) > 0 and div_amt > 0:
                        ex_close = ex_rows['close'].iloc[0]
                        if ex_close > div_amt:          # sanity check
                            factor = (ex_close - div_amt) / ex_close
                except (ValueError, TypeError):
                    skipped.append(f'{symbol} DIV {adate.date()} value={val}')

            if factor is not None and 0 < factor < 10:
                for col in ['open', 'high', 'low', 'close']:
                    prices_df.loc[before, f'adj_{col}'] *= factor
            elif factor is None and 'DIV' not in atype and 'DIVIDEND' not in atype:
                skipped.append(f'{symbol} {atype} {adate.date()} value={val}')

    if skipped:
        print("Skipped (could not parse):", skipped)

    return prices_df


prices_df = compute_adj_prices(prices_df, corp_df)

# ── Verification ─────────────────────────────────────────────────────────
# Any remaining >40% single-day move in adj_close is a data problem or missed action.
print("\nVerification — large single-day raw vs adjusted moves (>40%):")
print(f"  {'Symbol':>12}  {'Raw spikes':>10}  {'Adj spikes':>10}")
for sym in symbols:
    d = prices_df[prices_df['symbol'] == sym].sort_values('date')
    raw_big = (d['close'].pct_change().abs() > 0.40).sum()
    adj_big = (d['adj_close'].pct_change().abs() > 0.40).sum()
    flag = " ← check remaining" if adj_big > 0 else ""
    print(f"  {sym:>12}  {raw_big:>10}  {adj_big:>10}{flag}")

# Replace raw OHLC with adjusted so downstream cells see clean prices
prices_df['open']  = prices_df['adj_open']
prices_df['high']  = prices_df['adj_high']
prices_df['low']   = prices_df['adj_low']
prices_df['close'] = prices_df['adj_close']
prices_df = prices_df.drop(columns=['adj_open', 'adj_high', 'adj_low', 'adj_close'])
print("\nRaw OHLC replaced with adjusted prices.")

# ── NOW merge with company details ───────────────────────────────────────
# This must happen AFTER adjustment so combined_df gets adjusted prices, not raw.
combined_df = pd.merge(prices_df, companies_df, on='symbol', how='left')
print("Combined DataFrame after merging adjusted prices with companies:")
display(combined_df.head())


Verification — large single-day raw vs adjusted moves (>40%):
        Symbol  Raw spikes  Adj spikes
      RELIANCE           4           4 ← check remaining
      HDFCBANK           3           2 ← check remaining
    BHARTIARTL           1           0
          SBIN           1           0
     ICICIBANK           1           0
           TCS           3           0
    BAJFINANCE           2           3 ← check remaining
            LT           2           0
          LICI           1           1 ← check remaining
    HINDUNILVR           0           0

Raw OHLC replaced with adjusted prices.
Combined DataFrame after merging adjusted prices with companies:


,symbol,date,open,high,low,close,volume,company_name,sector,industry,cap_category,avg_mcap_cr
0,BAJFINANCE,2010-09-29,77.299314,79.776857,76.308298,76.764165,26000,Bajaj Finance Limited,Finance,Finance,Large Cap,608063.504856
1,BAJFINANCE,2010-09-30,76.803806,77.641215,75.936666,76.660109,17350,Bajaj Finance Limited,Finance,Finance,Large Cap,608063.504856
2,BAJFINANCE,2010-10-01,75.614586,79.013774,75.342056,76.798851,35941,Bajaj Finance Limited,Finance,Finance,Large Cap,608063.504856
3,BAJFINANCE,2010-10-04,78.092128,78.785840,76.803806,77.636260,31160,Bajaj Finance Limited,Finance,Finance,Large Cap,608063.504856
4,BAJFINANCE,2010-10-05,76.813716,78.092128,76.511456,77.200213,32141,Bajaj Finance Limited,Finance,Finance,Large Cap,608063.504856


In [17]:
# Fetch financial data
financials_query = f"""
SELECT symbol, year, quarter, period, revenue, net_profit, ebitda, eps, assets, liabilities, equity, debt, operating_cash_flow, free_cash_flow, book_value_per_share, operating_profit, ebit, shares_outstanding, cash_equivalents
FROM financials
WHERE symbol IN ({','.join([f"'{s}'" for s in symbols])})
ORDER BY symbol, year, quarter
"""
financials_df = conn.execute(financials_query).fetchdf()

# Convert 'year' and 'quarter' into a single 'date' column
# Assuming fiscal year starts in April (common in India), so FY 2023 means April 2023 - March 2024
def create_financial_date(row):
    try:
        year = int(row['year'])
        period_type = row['period']

        if period_type == 'annual':
            # For annual reports, assume fiscal year end (March 31 of next calendar year)
            return pd.to_datetime(f"{year + 1}-03-31")
        elif period_type == 'Q1':
            # Q1 (April-June) of the fiscal year
            return pd.to_datetime(f"{year}-06-30")
        elif period_type == 'Q2':
            # Q2 (July-September) of the fiscal year
            return pd.to_datetime(f"{year}-09-30")
        elif period_type == 'Q3':
            # Q3 (October-December) of the fiscal year
            return pd.to_datetime(f"{year}-12-31")
        elif period_type == 'Q4':
            # Q4 (January-March) of the fiscal year, falls in the next calendar year
            return pd.to_datetime(f"{year + 1}-03-31")
        else:
            return pd.NaT
    except ValueError:
        return pd.NaT

financials_df['date'] = financials_df.apply(create_financial_date, axis=1)

# Drop the original 'year', 'quarter', and 'period' columns as they are no longer needed
financials_df = financials_df.drop(columns=['year', 'quarter', 'period'])

# Fetch shareholding data
shareholding_query = f"""
SELECT symbol, quarter_end, promoter_pct, fii_pct, dii_pct, public_pct
FROM shareholding
WHERE symbol IN ({','.join([f"'{s}'" for s in symbols])})
ORDER BY symbol, quarter_end
"""
shareholding_df = conn.execute(shareholding_query).fetchdf()
shareholding_df['quarter_end'] = pd.to_datetime(shareholding_df['quarter_end'])

# Fetch company betas
betas_query = f"""
SELECT symbol, blended_beta
FROM company_betas
WHERE symbol IN ({','.join([f"'{s}'" for s in symbols])})
"""
company_betas_df = conn.execute(betas_query).fetchdf()

# Ensure all date columns have the same precision (nanoseconds) for merge_asof
combined_df['date'] = combined_df['date'].astype('datetime64[ns]')
financials_df['date'] = financials_df['date'].astype('datetime64[ns]')
shareholding_df['quarter_end'] = shareholding_df['quarter_end'].astype('datetime64[ns]')

# Drop rows with NaT in the 'date' column of combined_df and financials_df, 
# and NaN in 'symbol' column to ensure clean keys for sorting and merging
combined_df = combined_df.dropna(subset=['date', 'symbol'])
financials_df = financials_df.dropna(subset=['date', 'symbol'])

# Ensure unique (symbol, date) pairs in combined_df and financials_df before sorting
# This is a defensive measure to prevent potential issues with merge_asof's strict sorting checks
combined_df = combined_df.drop_duplicates(subset=['symbol', 'date'])
financials_df = financials_df.drop_duplicates(subset=['symbol', 'date'])

# merge_asof requires the 'on' key to be sorted globally (not per-symbol).
# Sorting by ['symbol', 'date'] causes dates to reset for each symbol, breaking global order.
combined_df = combined_df.sort_values(by='date').reset_index(drop=True)
financials_df = financials_df.sort_values(by='date').reset_index(drop=True)
shareholding_df = shareholding_df.sort_values(by='quarter_end').reset_index(drop=True)

# Merge financials using merge_asof
combined_df = pd.merge_asof(combined_df, financials_df, on='date', by='symbol', direction='backward')

# Merge shareholding using merge_asof
# Rename 'quarter_end' to 'date' temporarily for merge_asof
shareholding_df_temp = shareholding_df.rename(columns={'quarter_end': 'date'})
combined_df = pd.merge_asof(combined_df, shareholding_df_temp, on='date', by='symbol', direction='backward')

# Merge company betas (these are usually static or updated infrequently, so a simple merge is fine)
combined_df = pd.merge(combined_df, company_betas_df, on='symbol', how='left')

print("Combined DataFrame after merging financials, shareholding, and betas:")
display(combined_df.head())

Combined DataFrame after merging financials, shareholding, and betas:


,symbol,date,open,high,low,close,volume,company_name,sector,industry,...,book_value_per_share,operating_profit,ebit,shares_outstanding,cash_equivalents,promoter_pct,fii_pct,dii_pct,public_pct,blended_beta
0,SBIN,1996-04-01,19.801905,20.921143,19.801905,20.813524,3787050,State Bank of India,Banks,Banks,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.831
1,HDFCBANK,1996-04-01,1.532893,1.558875,1.530531,1.554151,96300,HDFC Bank Limited,Banks,Banks,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.791
2,RELIANCE,1996-04-01,12.209097,12.311941,12.106252,12.270803,8836850,Reliance Industries Limited,"Oil, Gas & Consumable Fuels",Oil,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.978
3,RELIANCE,1996-04-02,12.297249,12.458862,12.179713,12.303126,13118200,Reliance Industries Limited,"Oil, Gas & Consumable Fuels",Oil,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.978
4,SBIN,1996-04-02,21.097639,21.782096,20.546629,21.472153,5067850,State Bank of India,Banks,Banks,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.831


In [18]:
# Fetch macro data
macro_query = """
SELECT date, metric, value
FROM macro_data
ORDER BY date
"""
macro_df = conn.execute(macro_query).fetchdf()
macro_df['date'] = pd.to_datetime(macro_df['date']).astype('datetime64[ns]')

# Pivot macro_df to have metrics as columns
macro_pivot_df = macro_df.pivot_table(index='date', columns='metric', values='value').reset_index()
macro_pivot_df.columns.name = None
macro_pivot_df = macro_pivot_df.sort_values(by='date').reset_index(drop=True)

# Use merge_asof so each row gets the most recent macro reading (macro data is monthly/quarterly)
combined_df = combined_df.sort_values(by='date').reset_index(drop=True)
combined_df = pd.merge_asof(combined_df, macro_pivot_df, on='date', direction='backward')

print("Combined DataFrame after merging macro data:")
display(combined_df.head())

Combined DataFrame after merging macro data:


,symbol,date,open,high,low,close,volume,company_name,sector,industry,...,us_cpi_index,us_cpi_yoy,us_fed_rate_pct,us_gdp_growth_pct,us_gdp_usd_bn,us_real_interest_rate,us_unemployment_pct,us_unemployment_pct_m,usd_inr,wti_crude_usd
0,SBIN,1996-04-01,19.801905,20.921143,19.801905,20.813524,3787050,State Bank of India,Banks,Banks,...,155.5,NaN,5.31,NaN,NaN,NaN,NaN,5.5,NaN,NaN
1,HDFCBANK,1996-04-01,1.532893,1.558875,1.530531,1.554151,96300,HDFC Bank Limited,Banks,Banks,...,155.5,NaN,5.31,NaN,NaN,NaN,NaN,5.5,NaN,NaN
2,RELIANCE,1996-04-01,12.209097,12.311941,12.106252,12.270803,8836850,Reliance Industries Limited,"Oil, Gas & Consumable Fuels",Oil,...,155.5,NaN,5.31,NaN,NaN,NaN,NaN,5.5,NaN,NaN
3,RELIANCE,1996-04-02,12.297249,12.458862,12.179713,12.303126,13118200,Reliance Industries Limited,"Oil, Gas & Consumable Fuels",Oil,...,155.5,NaN,5.31,NaN,NaN,NaN,NaN,5.5,NaN,NaN
4,SBIN,1996-04-02,21.097639,21.782096,20.546629,21.472153,5067850,State Bank of India,Banks,Banks,...,155.5,NaN,5.31,NaN,NaN,NaN,NaN,5.5,NaN,NaN


In [19]:
import numpy as np

financial_cols = ['revenue', 'net_profit', 'equity', 'debt', 'ebitda', 'operating_cash_flow', 'book_value_per_share']

# Forward-fill per symbol so each day carries the last known quarterly value
combined_df = combined_df.sort_values(by=['symbol', 'date'])
combined_df[financial_cols] = combined_df.groupby('symbol')[financial_cols].ffill()

# profit_margin = net_profit / revenue
combined_df['profit_margin'] = combined_df['net_profit'] / combined_df['revenue']

# roe = net_profit / equity
combined_df['roe'] = combined_df['net_profit'] / combined_df['equity']

# debt_to_equity = debt / equity
combined_df['debt_to_equity'] = combined_df['debt'] / combined_df['equity']

# ebitda_margin = ebitda / revenue
combined_df['ebitda_margin'] = combined_df['ebitda'] / combined_df['revenue']

# book_to_price = book_value_per_share / close
combined_df['book_to_price'] = combined_df['book_value_per_share'] / combined_df['close']

# cash_flow_margin = operating_cash_flow / revenue
combined_df['cash_flow_margin'] = combined_df['operating_cash_flow'] / combined_df['revenue']

# Replace any inf values from division by zero with NaN
combined_df.replace([np.inf, -np.inf], np.nan, inplace=True)

print("Combined DataFrame after calculating derived financial features:")
display(combined_df.head())

Combined DataFrame after calculating derived financial features:


,symbol,date,open,high,low,close,volume,company_name,sector,industry,...,us_unemployment_pct,us_unemployment_pct_m,usd_inr,wti_crude_usd,profit_margin,roe,debt_to_equity,ebitda_margin,book_to_price,cash_flow_margin
18920,BAJFINANCE,2010-09-29,77.299314,79.776857,76.308298,76.764165,26000,Bajaj Finance Limited,Finance,Finance,...,NaN,NaN,45.050,77.86,NaN,NaN,NaN,NaN,NaN,NaN
18933,BAJFINANCE,2010-09-30,76.803806,77.641215,75.936666,76.660109,17350,Bajaj Finance Limited,Finance,Finance,...,NaN,9.5,44.690,79.97,NaN,NaN,NaN,NaN,NaN,NaN
18940,BAJFINANCE,2010-10-01,75.614586,79.013774,75.342056,76.798851,35941,Bajaj Finance Limited,Finance,Finance,...,NaN,NaN,44.563,81.58,NaN,NaN,NaN,NaN,NaN,NaN
18951,BAJFINANCE,2010-10-04,78.092128,78.785840,76.803806,77.636260,31160,Bajaj Finance Limited,Finance,Finance,...,NaN,NaN,44.413,81.47,NaN,NaN,NaN,NaN,NaN,NaN
18954,BAJFINANCE,2010-10-05,76.813716,78.092128,76.511456,77.200213,32141,Bajaj Finance Limited,Finance,Finance,...,NaN,NaN,44.450,82.82,NaN,NaN,NaN,NaN,NaN,NaN


In [20]:
# Ensure data is sorted by symbol and date for correct rolling calculations
combined_df = combined_df.sort_values(by=['symbol', 'date'])
combined_df['volume'] = combined_df['volume'].astype(float)

# Calculate daily_return
combined_df['daily_return'] = combined_df.groupby('symbol')['close'].pct_change()

# Calculate n-day returns
for days in [5, 20]:
    combined_df[f'{days}d_return'] = combined_df.groupby('symbol')['close'].pct_change(periods=days)

# Calculate Moving Averages (MA)
for days in [50, 200]:
    combined_df[f'{days}d_ma'] = combined_df.groupby('symbol')['close'].rolling(window=days).mean().reset_index(level=0, drop=True)

# Calculate 20d_volatility (rolling standard deviation of daily returns)
combined_df['20d_volatility'] = combined_df.groupby('symbol')['daily_return'].rolling(window=20).std().reset_index(level=0, drop=True)

# volume_ratio (e.g., Volume / 20-day Average Volume)
combined_df['20d_avg_volume'] = combined_df.groupby('symbol')['volume'].rolling(window=20).mean().reset_index(level=0, drop=True)
combined_df['volume_ratio'] = combined_df['volume'] / combined_df['20d_avg_volume']

# Display the final DataFrame with all features
print("Final DataFrame with all calculated features:")
display(combined_df.head())

Final DataFrame with all calculated features:


,symbol,date,open,high,low,close,volume,company_name,sector,industry,...,book_to_price,cash_flow_margin,daily_return,5d_return,20d_return,50d_ma,200d_ma,20d_volatility,20d_avg_volume,volume_ratio
18920,BAJFINANCE,2010-09-29,77.299314,79.776857,76.308298,76.764165,26000.0,Bajaj Finance Limited,Finance,Finance,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
18933,BAJFINANCE,2010-09-30,76.803806,77.641215,75.936666,76.660109,17350.0,Bajaj Finance Limited,Finance,Finance,...,NaN,NaN,-0.001356,NaN,NaN,NaN,NaN,NaN,NaN,NaN
18940,BAJFINANCE,2010-10-01,75.614586,79.013774,75.342056,76.798851,35941.0,Bajaj Finance Limited,Finance,Finance,...,NaN,NaN,0.001810,NaN,NaN,NaN,NaN,NaN,NaN,NaN
18951,BAJFINANCE,2010-10-04,78.092128,78.785840,76.803806,77.636260,31160.0,Bajaj Finance Limited,Finance,Finance,...,NaN,NaN,0.010904,NaN,NaN,NaN,NaN,NaN,NaN,NaN
18954,BAJFINANCE,2010-10-05,76.813716,78.092128,76.511456,77.200213,32141.0,Bajaj Finance Limited,Finance,Finance,...,NaN,NaN,-0.005617,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [21]:
# ── Corporate action features ─────────────────────────────────────────────
# Add is_ex_dividend, dividend_amount, is_split, split_factor as model features.
# These are separate from the price adjustment done earlier — here we mark the
# event days so the model can learn that these dates carry special significance.

div_df = corp_df[corp_df['action_type'].str.upper().str.contains('DIV')][['symbol', 'date', 'value']].copy()
div_df = div_df.rename(columns={'value': 'dividend_amount'})
div_df['is_ex_dividend'] = 1
div_df['dividend_amount'] = pd.to_numeric(div_df['dividend_amount'], errors='coerce').fillna(0.0)

split_df = corp_df[corp_df['action_type'].str.upper().str.contains('SPLIT|BONUS')][['symbol', 'date', 'value']].copy()

def parse_split_factor(val):
    try:
        if isinstance(val, str) and ':' in val:
            a, b = val.split(':', 1)
            return float(a.strip()) / float(b.strip())
        return float(val)
    except (ValueError, ZeroDivisionError, AttributeError):
        return 0.0

split_df['split_factor'] = split_df['value'].apply(parse_split_factor)
split_df['is_split'] = 1
split_df = split_df.drop(columns=['value'])

combined_df = combined_df.merge(
    div_df[['symbol', 'date', 'is_ex_dividend', 'dividend_amount']],
    on=['symbol', 'date'], how='left'
)
combined_df = combined_df.merge(
    split_df[['symbol', 'date', 'is_split', 'split_factor']],
    on=['symbol', 'date'], how='left'
)

combined_df['is_ex_dividend'] = combined_df['is_ex_dividend'].fillna(0).astype(int)
combined_df['dividend_amount'] = combined_df['dividend_amount'].fillna(0.0)
combined_df['is_split']        = combined_df['is_split'].fillna(0).astype(int)
combined_df['split_factor']    = combined_df['split_factor'].fillna(0.0)

print(f"Ex-dividend days : {combined_df['is_ex_dividend'].sum()}")
print(f"Split days       : {combined_df['is_split'].sum()}")
print(combined_df[['symbol', 'date', 'is_ex_dividend', 'dividend_amount', 'is_split', 'split_factor']][
    (combined_df['is_ex_dividend'] == 1) | (combined_df['is_split'] == 1)
].head(10))

Ex-dividend days : 306
Split days       : 20
          symbol       date  is_ex_dividend  dividend_amount  is_split  \
188   BAJFINANCE 2011-06-29               1         0.097158         0   
440   BAJFINANCE 2012-07-05               1         0.116590         0   
687   BAJFINANCE 2013-07-04               1         0.150000         0   
932   BAJFINANCE 2014-07-03               1         0.160000         0   
1179  BAJFINANCE 2015-07-09               1         0.180000         0   
1349  BAJFINANCE 2016-03-16               1         0.180000         0   
1429  BAJFINANCE 2016-07-14               1         0.070000         0   
1467  BAJFINANCE 2016-09-08               0         0.000000         1   
1670  BAJFINANCE 2017-07-06               1         0.360000         0   
1919  BAJFINANCE 2018-07-05               1         0.400000         0   

      split_factor  
188            0.0  
440            0.0  
687            0.0  
932            0.0  
1179           0.0  
1349          

In [22]:
# --- NaN Handling ---

combined_df = combined_df[combined_df['200d_ma'].notna()].reset_index(drop=True)


# Step 2: Forward-fill macro and financial ratio columns per symbol
macro_cols = [c for c in combined_df.columns if c.startswith('us_') or c in ['usd_inr', 'wti_crude_usd']]
financial_ratio_cols = ['profit_margin', 'roe', 'debt_to_equity', 'ebitda_margin', 'book_to_price', 'cash_flow_margin']
fill_cols = macro_cols + financial_ratio_cols
combined_df[fill_cols] = combined_df.groupby('symbol')[fill_cols].transform('ffill')

# Step 3: Drop any rows still NaN in critical columns
critical_cols = ['daily_return', '5d_return', '20d_return', '50d_ma', '200d_ma', '20d_volatility', 'volume_ratio']
combined_df = combined_df.dropna(subset=critical_cols).reset_index(drop=True)

print("Shape after NaN handling:", combined_df.shape)
print("\nRemaining NaN counts:")
nan_counts = combined_df.isna().sum()
print(nan_counts[nan_counts > 0])

Shape after NaN handling: (52150, 71)

Remaining NaN counts:
revenue                  28114
net_profit               28114
ebitda                   28114
eps                      28134
assets                   28114
liabilities              28114
equity                   28114
debt                     28114
operating_cash_flow      28114
free_cash_flow           28361
book_value_per_share     28134
operating_profit         28114
ebit                     38912
shares_outstanding       28134
cash_equivalents         29349
promoter_pct             45168
fii_pct                  45168
dii_pct                  45168
public_pct               45168
brent_crude_usd          12917
bse_sensex                 549
gold_inr                  8088
gold_usd                  4724
india_cpi_yoy            51968
india_gdp_growth_pct     51968
india_gdp_usd_bn         51968
india_repo_rate_pct      50502
india_vix                12596
india_wpi_proxy_pct      51968
nifty50                  11753
us_cpi_in

In [23]:
combined_df.to_parquet("C:\\Users\\Keerti A M\\Documents\\New folder\\data\\combined_features.parquet", index=False)
print("Saved successfully. Shape:", combined_df.shape)

Saved successfully. Shape: (52150, 71)
